# Escolaridad e internet por perfil (Estadísticas Censales a Escalas Geoelectorales 2020)

El CCPC no trae escolaridad. Se toma del producto conjunto INEGI-INE **Estadísticas Censales a
Escalas Geoelectorales 2020** (ECEG): indicadores del Censo de Población y Vivienda 2020 asignados
a cada sección electoral del marco seccional del INE con corte de enero de 2021.

- Variable principal: grado promedio de escolaridad de la población de 15 años y más (`GRAPROES`),
  por sexo (`GRAPROES_F`, `GRAPROES_M`).
- Variable complementaria: % de la población de 18 años y más con educación posbásica (`P18YM_PB`).
- % de viviendas particulares habitadas con internet (`VPH_INTER / VIVPARH_CV`), util para elegir
  canales de campaña. Es dato de la vivienda: igual para ambos sexos.
- Cada celda (sección x sexo x grupo de edad) recibe el valor de su sección para su sexo.

Salidas en `DATA/PREPROCESSED/`: `secciones_eceg_2020.parquet`, `escolaridad_perfiles.csv`,
`escolaridad_subperfiles.csv`, `escolaridad_perfil_tiposec.csv`.

## 1. Configuración

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})

## 2. Parametros y rutas

Tramos de grado promedio: menos de 8 (por debajo de secundaria completa), 8 a 10 (secundaria),
10 a 12 (media superior incompleta) y 12 o más (media superior completa o más).

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DIR_PREPROCESSED = ROOT / "DATA" / "PREPROCESSED"
ARCHIVO_ECEG = ROOT / "DATA" / "RAW" / "eceg_2020" / "conjunto_de_datos" / "INE_SECCION_2020.csv"
LLAVES = ["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"]

COLUMNAS_ECEG = [
    "ENTIDAD", "SECCIÓN", "POBTOT", "P_15YMAS",
    "P_18YMAS", "P_18YMAS_F", "P_18YMAS_M",
    "GRAPROES", "GRAPROES_F", "GRAPROES_M",
    "P18YM_PB", "P18YM_PB_F", "P18YM_PB_M",
    "VIVPARH_CV", "VPH_INTER",
]
CORTES_ESC = [0, 8, 10, 12, 30]
TRAMOS_ESC = ["<8", "8-10", "10-12", "12+"]
COLORES_TRAMO = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
TIPOSEC_NOMBRE = {"U": "Urbana", "M": "Mixta", "R": "Rural"}

### 2.1 Funciones auxiliares

In [ ]:
def media_ponderada(df, col, por):
    grupos = [df[c] for c in por]
    wx = (df[col] * df["peso"]).groupby(grupos).sum()
    return wx / df["peso"].where(df[col].notna()).groupby(grupos).sum()


def distribucion(df, por, col):
    t = df.pivot_table(index=por, columns=col, values="peso", aggfunc="sum", observed=False)
    return t.div(t.sum(axis=1), axis=0) * 100


def corr_ponderada(x, y, w):
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    return np.average((x - mx) * (y - my), weights=w) / np.sqrt(np.average((x - mx) ** 2, weights=w) * np.average((y - my) ** 2, weights=w))


def resumen_escolaridad(df, por):
    out = pd.DataFrame({
        "personas": df.groupby(por)["peso"].sum(),
        "escolaridad": media_ponderada(df, "escolaridad", por),
        "pct_posbasica": media_ponderada(df, "pct_posbasica", por),
        "pct_internet": media_ponderada(df, "pct_internet", por),
    })
    dist = distribucion(df, por, "tramo_esc")
    dist.columns = [f"pct_esc_{c}" for c in dist.columns]
    return out.join(dist)

## 3. Carga de la ECEG por sección

El CSV viene en latin-1. Los valores negativos (`-6`) son el código de confidencialidad del INEGI y
las secciones sin población traen `0`; ambos se tratan como dato faltante.

In [ ]:
eceg = pd.read_csv(ARCHIVO_ECEG, encoding="latin1", usecols=COLUMNAS_ECEG)
eceg = eceg.rename(columns={"ENTIDAD": "EDOCVE"})
assert not eceg.duplicated(["EDOCVE", "SECCIÓN"]).any()

valores = eceg.columns.drop(["EDOCVE", "SECCIÓN"])
print("Valores negativos por columna:", (eceg[valores] < 0).sum()[lambda s: s > 0].to_dict())
eceg[valores] = eceg[valores].mask(eceg[valores] < 0)
sin_poblacion = eceg["P_15YMAS"].fillna(0) == 0
eceg.loc[sin_poblacion, ["GRAPROES", "GRAPROES_F", "GRAPROES_M"]] = np.nan
print(f"Secciones: {len(eceg):,}  sin población de 15+: {sin_poblacion.sum()}")

### 3.1 Indicadores por sección y sexo

Porcentaje con educación posbásica sobre la población de 18 años y más del mismo sexo, y porcentaje
de viviendas particulares habitadas con internet.

In [ ]:
for sufijo in ["", "_F", "_M"]:
    eceg[f"PCT_POSBASICA{sufijo}"] = eceg[f"P18YM_PB{sufijo}"] / eceg[f"P_18YMAS{sufijo}"].where(eceg[f"P_18YMAS{sufijo}"] > 0) * 100

eceg["PCT_INTERNET"] = eceg["VPH_INTER"] / eceg["VIVPARH_CV"].where(eceg["VIVPARH_CV"] > 0) * 100

eceg[["GRAPROES", "GRAPROES_F", "GRAPROES_M", "PCT_POSBASICA", "PCT_INTERNET"]].describe().round(2)

### 3.2 Guardar tabla de secciones

In [ ]:
secciones_eceg = eceg[["EDOCVE", "SECCIÓN", "POBTOT", "P_15YMAS", "P_18YMAS",
                       "GRAPROES", "GRAPROES_F", "GRAPROES_M",
                       "PCT_POSBASICA", "PCT_POSBASICA_F", "PCT_POSBASICA_M", "PCT_INTERNET"]]
secciones_eceg.to_parquet(DIR_PREPROCESSED / "secciones_eceg_2020.parquet", index=False)
print(f"secciones_eceg_2020.parquet: {len(secciones_eceg):,} filas")

## 4. Union con las celdas

Celdas con perfil y subperfil del notebook 09 (clases latentes) y tipo de sección del 03.
Cada celda toma la escolaridad de su sección para su sexo; el sexo no binario (solo 2024 y sin
celdas con historial completo) usaria el total.

In [ ]:
celdas = pd.read_parquet(DIR_PREPROCESSED / "celdas_subperfiles.parquet")
tipo = pd.read_parquet(DIR_PREPROCESSED / "celdas_historial.parquet", columns=LLAVES + ["TIPOSEC"])
celdas["GRUPO_EDAD"] = celdas["GRUPO_EDAD"].astype(str)
tipo["GRUPO_EDAD"] = tipo["GRUPO_EDAD"].astype(str)
celdas = celdas.merge(tipo, on=LLAVES, how="left")
celdas["EDOCVE"] = celdas["EDOCVE"].astype(int)

celdas = celdas.merge(secciones_eceg, on=["EDOCVE", "SECCIÓN"], how="left", indicator=True)
union = celdas["_merge"] == "both"
print(f"Celdas con sección en la ECEG: {union.mean():.2%}  personas: {celdas.loc[union, 'peso'].sum() / celdas['peso'].sum():.2%}")
celdas = celdas.drop(columns="_merge")

### 4.1 Escolaridad de la celda segun su sexo

In [ ]:
sexo = celdas["SEXO"]
celdas["escolaridad"] = np.select([sexo == 1, sexo == 0], [celdas["GRAPROES_F"], celdas["GRAPROES_M"]], celdas["GRAPROES"])
celdas["pct_posbasica"] = np.select([sexo == 1, sexo == 0], [celdas["PCT_POSBASICA_F"], celdas["PCT_POSBASICA_M"]], celdas["PCT_POSBASICA"])
celdas["pct_internet"] = celdas["PCT_INTERNET"]
celdas["tramo_esc"] = pd.cut(celdas["escolaridad"], CORTES_ESC, labels=TRAMOS_ESC, right=False)

sin_dato = celdas["escolaridad"].isna()
print(f"Personas sin escolaridad (secciones confidenciales o vacias): {celdas.loc[sin_dato, 'peso'].sum():,}")
nacional = np.average(celdas.loc[~sin_dato, "escolaridad"], weights=celdas.loc[~sin_dato, "peso"])
print(f"Grado promedio ponderado por votantes: {nacional:.2f}")

## 5. Escolaridad y participación por sección

Antes de caracterizar perfiles: participación promedio 2009-2024 de las secciones del 03 segun su
grado promedio de escolaridad y tipo de sección, ponderada por lista nominal 2024.

In [ ]:
secciones = pd.read_parquet(DIR_PREPROCESSED / "secciones_historial.parquet", columns=["EDOCVE", "SECCIÓN", "promedio", "TIPOSEC", "LN2024"])
secciones["EDOCVE"] = secciones["EDOCVE"].astype(int)
secciones = secciones.merge(secciones_eceg[["EDOCVE", "SECCIÓN", "GRAPROES"]], on=["EDOCVE", "SECCIÓN"]).dropna(subset=["GRAPROES"])
secciones["peso"] = secciones["LN2024"]
secciones["tramo_esc"] = pd.cut(secciones["GRAPROES"], CORTES_ESC, labels=TRAMOS_ESC, right=False)

print(f"Secciones: {len(secciones):,}  correlación ponderada escolaridad-participación: "
      f"{corr_ponderada(secciones['GRAPROES'], secciones['promedio'], secciones['peso']):.2f}")
for t, g in secciones.groupby("TIPOSEC"):
    print(f"  {TIPOSEC_NOMBRE[t]}: {corr_ponderada(g['GRAPROES'], g['promedio'], g['peso']):.2f}")

media_ponderada(secciones, "promedio", ["tramo_esc", "TIPOSEC"]).unstack().rename(columns=TIPOSEC_NOMBRE).round(1)

## 6. Escolaridad por perfil

In [ ]:
res_perfil = resumen_escolaridad(celdas, ["perfil"])
res_perfil.round(2)

### 6.1 Distribución por tramo de escolaridad

Porcentaje de las personas de cada perfil segun el grado promedio de su sección (para su sexo).

In [ ]:
dist = res_perfil[[f"pct_esc_{t}" for t in TRAMOS_ESC]]
fig, ax = plt.subplots(figsize=(10, 4.2))
izquierda = np.zeros(len(dist))
for tramo, color, oscuro in zip(TRAMOS_ESC, COLORES_TRAMO, [False, False, True, True]):
    v = dist[f"pct_esc_{tramo}"].to_numpy()
    ax.barh(dist.index.astype(str), v, left=izquierda, color=color, edgecolor="white", linewidth=2, label=tramo)
    for y, (x0, w) in enumerate(zip(izquierda, v)):
        if w >= 8:
            ax.text(x0 + w / 2, y, f"{w:.0f}", ha="center", va="center", fontsize=9, color="white" if oscuro else "#1f1f1f")
    izquierda += v
ax.set_yticks(range(len(dist)))
ax.set_yticklabels([f"Perfil {p}  ({res_perfil.loc[p, 'escolaridad']:.1f} grados)" for p in dist.index])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% de personas del perfil")
ax.set_title("Grado promedio de escolaridad de la sección, por perfil")
ax.legend(title="Grados", ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

### 6.2 Por perfil y tipo de sección

Separa el efecto territorial: un perfil con muchas secciones rurales puede tener menor escolaridad
promedio aunque sus secciones urbanas sean de escolaridad alta.

In [ ]:
res_tiposec = resumen_escolaridad(celdas, ["perfil", "TIPOSEC"])
tabla = res_tiposec["escolaridad"].unstack().rename(columns=TIPOSEC_NOMBRE)[["Urbana", "Mixta", "Rural"]]
pct_tipo = (res_tiposec["personas"] / res_tiposec.groupby(level="perfil")["personas"].transform("sum") * 100).unstack().rename(columns=TIPOSEC_NOMBRE)
tabla.round(2).join(pct_tipo[["Urbana", "Mixta", "Rural"]].round(1), rsuffix=" (% personas)")

## 7. Escolaridad por subperfil

In [ ]:
res_sub = resumen_escolaridad(celdas, ["perfil", "subperfil"])
res_sub["pct_perfil"] = res_sub["personas"] / res_sub.groupby(level="perfil")["personas"].transform("sum") * 100
res_sub.round(2)

## 8. Guardar resultados

In [ ]:
res_perfil.round(3).to_csv(DIR_PREPROCESSED / "escolaridad_perfiles.csv")
res_sub.round(3).to_csv(DIR_PREPROCESSED / "escolaridad_subperfiles.csv")
res_tiposec.round(3).to_csv(DIR_PREPROCESSED / "escolaridad_perfil_tiposec.csv")
print("Guardados: escolaridad_perfiles.csv, escolaridad_subperfiles.csv, escolaridad_perfil_tiposec.csv")

## 9. Limitaciones

- **Dato ecológico:** la escolaridad es el promedio de la sección para el sexo de la celda (15 años
  y más), no la de su grupo de edad ni la de las personas que votaron. Describe el entorno
  educativo de cada perfil, no la escolaridad individual.
- **Fechas distintas:** censo de marzo de 2020 contra lista nominal de 2024.
- **Marco seccional:** corte de enero de 2021; todas las secciones de las celdas existen en la ECEG.